# 08-1 空間流病：樓層翼區侵襲率與 Spot Map

長官問：「在哪裡最嚴重？」

松柏護理之家有 3 層樓 × 2 翼區（A / B），共 280 位住民。
我們要找出哪些區域侵襲率最高，並畫出 spot map。

流程：**資料準備 → floor × wing 侵襲率 → 熱力圖 → 每間房侵襲率 → Spot Map → 致死率空間比較**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 資料準備 ---
import pathlib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

print(f"住民數：{len(df)}")
print(f"感染：{df['infected'].sum()}")
print(f"死亡：{df['died'].sum()}")
print(f"\n樓層：{sorted(df['floor'].unique())}")
print(f"翼區：{sorted(df['wing'].unique())}")
print(f"房間數：{df['room'].nunique()}")

### 📌 Step 1 結果解讀

**輸出說明**

| 指標 | 數值 | 含義 |
|---|---|---|
| 住民數 | 280 | 全護理之家觀察對象 |
| 感染 | 121 | `clinical_severity != "not_ill"` 的人數 |
| 死亡 | 19 | `outcome == "dead"` 的人數 |

**旗標欄位（0/1 indicator）的用途**

- `df["infected"]` 每人是 0（未感染）或 1（感染）
- 對旗標欄做 `.sum()` = 計算感染人數；做 `.mean()` = 計算感染率（侵襲率）
- 這是流病計算的基本功，後面所有分析都靠它

> **小訣竅**：建立旗標後立刻用 `print` 確認加總是否合理（例如感染人數不應超過住民總數），這是避免後續分析錯誤的好習慣。

In [ ]:
# --- Step 2: Floor × Wing 侵襲率 ---
spatial = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
    died=("died", "sum"),
).reset_index()
spatial["attack_rate"] = (spatial["infected"] / spatial["total"] * 100).round(1)
spatial["cfr"] = (spatial["died"] / spatial["infected"] * 100).round(1)

print("=== 樓層翼區統計 ===")
print(spatial.to_string(index=False))

### 📌 Step 2 程式碼拆解 & 結果解讀

**`groupby().agg()` 白話拆解**

```
spatial = df.groupby(["floor", "wing"])   ← 把 280 筆資料依 floor + wing 分成 6 組
    .agg(
        total    = ("case_id",   "count"),  ← 每組的總人數（數 case_id 筆數）
        infected = ("infected",  "sum"),    ← 每組感染人數（旗標加總）
        died     = ("died",      "sum"),    ← 每組死亡人數
    )
    .reset_index()                          ← 讓 floor/wing 回到普通欄位（不是索引）
```

**`agg()` 語法公式**：`新欄位名 = ("來源欄位", "統計函數")`

常用統計函數：`"count"`（計筆數）、`"sum"`（加總）、`"mean"`（平均）、`"max"`、`"min"`

**結果解讀重點**

- `attack_rate`（侵襲率）：數值越高 → 那個翼區感染越嚴重
- `cfr`（致死率）：感染者中死亡的比例，反映病情嚴重度
- 比較侵襲率 vs 致死率：侵襲率高 ≠ 致死率高，兩者要分開看

> ⚠️ **常見陷阱**：只看絕對病例數（如「2F-A 有 24 人感染最多！」）而忽略分母。24/44（54.5%）和 25/50（50.0%）在人數上接近，但侵襲率差距不大，不能只憑數字大就下結論。

In [ ]:
# --- Step 3: 侵襲率熱力圖 ---
heatmap_ar = spatial.pivot(index="floor", columns="wing", values="attack_rate")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 侵襲率
sns.heatmap(heatmap_ar, annot=True, fmt=".1f", cmap="YlOrRd",
            cbar_kws={"label": "%"}, ax=axes[0])
axes[0].set_title("侵襲率 (%) by Floor \u00d7 Wing")
axes[0].set_ylabel("Floor")

# 致死率
heatmap_cfr = spatial.pivot(index="floor", columns="wing", values="cfr")
sns.heatmap(heatmap_cfr, annot=True, fmt=".1f", cmap="Reds",
            cbar_kws={"label": "%"}, ax=axes[1])
axes[1].set_title("致死率 (%) by Floor \u00d7 Wing")
axes[1].set_ylabel("Floor")

plt.tight_layout()
plt.show()

print("\u2192 侵襲率最高的區域：")
top = spatial.nlargest(3, "attack_rate")
for _, row in top.iterrows():
    print(f"  {row['floor']}F-{row['wing']} 翼：{row['attack_rate']}%")

### 📌 Step 3 程式碼拆解 & 熱力圖解讀

**`pivot()` 的作用**

`groupby().agg()` 產出的是「長表」（每行是一個 floor-wing 組合）：
```
floor  wing  attack_rate
1      A     34.1
1      B     21.3
...
```

`pivot(index="floor", columns="wing", values="attack_rate")` 把它變成「矩陣」：
```
wing    A     B
floor
1      34.1  21.3
2      54.5  50.0
3      41.7  57.4
```

`sns.heatmap()` 需要矩陣格式才能畫格子，**`pivot()` 是必要的前處理步驟**。

**`heatmap` 參數說明**

| 參數 | 意思 |
|---|---|
| `annot=True` | 在每個格子裡顯示數值（不然只有顏色） |
| `fmt=".1f"` | 數值格式：一位小數 |
| `cmap="YlOrRd"` | 色板：黃→橘→紅，直覺對應低→中→高侵襲率 |
| `cmap="Reds"` | 致死率用純紅漸層，與侵襲率圖視覺區分 |

**熱力圖解讀**

- 顏色越深（越紅）= 侵襲率 / 致死率越高
- 看**行**（同一樓層 A vs B 翼）：是否某翼區系統性偏高？
- 看**列**（同一翼區不同樓層）：是否 2F/3F 比 1F 嚴重？
- 侵襲率高的翼區 + 致死率也高 → 該區域可能有特別危險的暴露

> **何時用熱力圖？** 分析對象剛好有**兩個分類維度**（floor + wing）時，熱力圖最直覺。如果有三個維度，就需要考慮分面（facet）或互動式圖表。

In [ ]:
# --- Step 4: 每間房侵襲率 ---
room_stats = df.groupby("room").agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
).reset_index()
room_stats["attack_rate"] = (room_stats["infected"] / room_stats["total"] * 100).round(1)

# 解析 room 名稱（例如 "2A-03" → floor=2, wing=A, room_num=3）
room_stats["floor_num"] = room_stats["room"].str[0].astype(int)
room_stats["wing_code"] = room_stats["room"].str[1]
room_stats["room_num"] = room_stats["room"].str.split("-").str[1].astype(int)

print(f"共 {len(room_stats)} 間房")
print(f"\n侵襲率 100% 的房間（全部住民都感染）：")
full = room_stats[room_stats["attack_rate"] == 100.0]
print(f"  共 {len(full)} 間")
print(f"\n侵襲率 0% 的房間（無人感染）：")
zero = room_stats[room_stats["attack_rate"] == 0.0]
print(f"  共 {len(zero)} 間")

### 📌 Step 4 程式碼拆解 & 結果解讀

**字串解析拆解**

房間代號格式：`"2A-03"`（樓層 + 翼區代碼 + 連字號 + 房間號）

| 操作 | 以 "2A-03" 為例 | 說明 |
|---|---|---|
| `room.str[0].astype(int)` | `"2"` → `2` | 取第 0 個字元 = 樓層號 |
| `room.str[1]` | `"A"` | 取第 1 個字元 = 翼區代碼 |
| `room.str.split("-").str[1].astype(int)` | `"03"` → `3` | 以 `-` 切割，取第二段 = 房間號 |

**為什麼需要解析成數字？**

`scatter()` 的 x/y 需要**數值**才能定位座標。字串 `"2A-03"` 不能直接放在數線上，解析後的整數 `2`、`3` 才能。

**結果解讀**

- 侵襲率 100% 的房間：整房所有住民都感染 → 但需注意這些房間是 1 人房還是多人房？
  - 1 人感染 1 人 = 100%，與 5 人感染 5 人 = 100% 的流行病學意義完全不同
- 侵襲率 0% 的房間：全員倖免 → 是否有保護性因素？（如不使用公共熱水設施）

> 小房間（1-2 人）的侵襲率很不穩定，100% 可能只是 1/1 的偶然，不能過度解讀。Step 5 的 Spot Map 圓點大小就是為了可視化這個問題。

In [ ]:
# --- Step 5: Spot Map（模擬護理之家平面圖）---
# X 軸 = 房間號碼，A 翼在左半、B 翼在右半
# Y 軸 = 樓層
max_room_a = room_stats[room_stats["wing_code"] == "A"]["room_num"].max()
gap = 5  # A 翼和 B 翼之間的間距

room_stats["x"] = room_stats.apply(
    lambda r: r["room_num"] if r["wing_code"] == "A"
    else r["room_num"] + max_room_a + gap,
    axis=1,
)

fig, ax = plt.subplots(figsize=(14, 5))
sc = ax.scatter(
    room_stats["x"],
    room_stats["floor_num"],
    s=room_stats["total"] * 50,
    c=room_stats["attack_rate"],
    cmap="YlOrRd",
    edgecolors="black",
    linewidth=0.5,
    alpha=0.8,
    vmin=0,
    vmax=100,
)
plt.colorbar(sc, label="侵襲率 (%)")

# 分隔線與標籤
mid_x = max_room_a + gap / 2
ax.axvline(x=mid_x, color="gray", linestyle="--", alpha=0.5)
ax.text(max_room_a / 2, 3.5, "A 翼", ha="center", fontsize=12, fontweight="bold")
ax.text(max_room_a + gap + 12, 3.5, "B 翼", ha="center", fontsize=12, fontweight="bold")

ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["1F", "2F", "3F"])
ax.set_xlabel("房間號碼")
ax.set_ylabel("樓層")
ax.set_title("Spot Map \u2014 每間房的侵襲率（圓點大小 = 住民數，顏色 = 侵襲率）")
plt.tight_layout()
plt.show()

print("\u2192 深色 (高侵襲率) 圓點是否集中在某些翼區？")
print("\u2192 這些高風險區域可能共用汙染的熱水管線或蓮蓬頭")

### 📌 Step 5 程式碼拆解 & Spot Map 解讀

**x 座標偏移設計**

A 翼和 B 翼的房間號都從 01 開始。若直接用 `room_num` 當 x 座標，兩翼的圓點會疊在一起。

解法：
```
B 翼的 x = B 翼房間號 + A 翼最大房間號 + gap(=5)
```

例如 A 翼有 15 間房（room_num 1~15），B 翼 room 1 的 x = 1 + 15 + 5 = 21。
這樣 A 翼佔 x ∈ [1,15]，分隔空白在 [16,20]，B 翼從 x=21 開始，模擬真實平面圖的左右佈局。

> 用 `max_room_a + gap` 而非寫死數字 30，是因為如果資料改變（A 翼房間數變了），程式碼自動調整，不需要手動修改。

**scatter 參數說明**

| 參數 | 代表什麼 |
|---|---|
| `s = total * 50` | 圓點**面積**（s = size），乘以 50 是視覺縮放係數 |
| `c = attack_rate` | 圓點顏色對應侵襲率 |
| `cmap="YlOrRd"` | 色板同熱力圖，保持視覺一致 |
| `vmin=0, vmax=100` | 固定色軸範圍 0-100%（不然不同圖的顏色基準不一致）|
| `alpha=0.8` | 透明度 0.8，讓重疊圓點仍可見 |

**Spot Map 解讀（四個問題）**

1. **哪裡最紅？** → 深色聚集區 = 高風險房間群，優先調查水源
2. **大圓 vs 小圓？** → 大圓侵襲率更可信；小圓（1人房 100%）只是偶然，不可過度解讀
3. **A 翼 vs B 翼** → 分隔線兩側顏色差異大？→ 翼區特異性暴露（管線不同？）
4. **樓層高低** → 2F/3F 普遍比 1F 深？→ 可能垂直管線有問題

> **熱力圖 vs Spot Map 選哪個？**
> - 熱力圖：看 6 個 floor×wing 組合的整體摘要，適合報告和決策
> - Spot Map：看每間房的細節分布，找異常聚集點，適合深入調查

In [ ]:
# --- Step 6: 翼區侵襲率排序條圖 ---
spatial["label"] = spatial["floor"].astype(str) + "F-" + spatial["wing"]
spatial_sorted = spatial.sort_values("attack_rate", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(
    spatial_sorted["label"],
    spatial_sorted["attack_rate"],
    color=["#e34a33" if ar > 50 else "#2c7fb8" for ar in spatial_sorted["attack_rate"]],
)
for bar, val in zip(bars, spatial_sorted["attack_rate"]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{val}%", va="center")

ax.set_xlabel("侵襲率 (%)")
ax.set_title("各翼區侵襲率（紅色 > 50%）")
ax.set_xlim(0, 70)
plt.tight_layout()
plt.show()

print("\u2192 侵襲率超過 50% 的翼區需要優先進行環境採檢")

### 📌 Step 6 解讀 & 下一步

**條圖解讀**

- 條越長、顏色越紅 → 侵襲率越高、越需要優先行動
- 紅色（>50%）代表**超過一半的住民感染**，是高度警戒的訊號
- 排序讓你一眼看出相對風險：最右側的翼區最危險

**看到翼區差異後，下一步**

| 問題 | 分析方法 |
|---|---|
| 哪裡死亡率也高？ | 比較左邊的致死率熱力圖 |
| 淋浴使用率是否集中在高侵襲率翼區？ | `groupby("wing").agg(shower_pct=("shower_use","mean"))` |
| 高風險區域共用什麼基礎設施？ | 對照護理之家工程圖，標記熱水管線走向 |
| 有沒有空間群聚的統計顯著性？ | 進階：Moran's I 空間自相關（本章範圍外）|

**結論句型**

> 「3F-B 翼侵襲率最高（57.4%），其次為 2F-A（54.5%）和 2F-B（50.0%）。這三個翼區應優先進行熱水系統環境採樣，並調查其共用管線情況。」

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| floor × wing 侵襲率 | `groupby().agg()` 多指標計算 |
| 熱力圖 | `sns.heatmap()` + `pivot()` |
| 每間房分析 | 字串解析 `room` 欄位 |
| Spot Map | `scatter()` 大小=住民、顏色=侵襲率 |
| 排序條圖 | 用顏色標記高風險區域 |

**結論**：2F 和 3F-B 翼侵襲率最高，這些區域可能共用被汙染的熱水系統。
下一個 notebook（`08_spatial_choropleth`）示範地理 choropleth 的概念。